Se extrae y limpian canciones para crear archivo "ITEMS.CSV"

# Importaciones

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', None)

# Ruta flexible para poder ejecutar el notebook desde distintas carpetas.
RAW_PATH = Path('data/raw/dataset.csv')
if not RAW_PATH.exists():
	RAW_PATH = Path('../../data/raw/dataset.csv')
 
OUTPUT_PATH = Path('/data/processed/items.csv')
if not OUTPUT_PATH.exists():
	OUTPUT_PATH = Path('../../../data/processed/items.csv')

# Carga de Dataset Original

In [2]:
df_raw = pd.read_csv(RAW_PATH)
print(f'Shape original: {df_raw.shape}')
df_raw.head(3)

Shape original: (114000, 21)


,Unnamed: 0,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,0.461,1,-6.746,0,0.1430,0.0322,0.000001,0.358,0.715,87.917,4,acoustic
1,1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,False,0.420,0.166,1,-17.235,1,0.0763,0.9240,0.000006,0.101,0.267,77.489,4,acoustic
2,2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,False,0.438,0.359,0,-9.734,1,0.0557,0.2100,0.000000,0.117,0.120,76.332,4,acoustic


# Normalización

In [4]:
# Verificar rango de features de audio (deben estar entre 0 y 1, excepto loudness y tempo)

df = df_raw.copy()

audio_features = ['danceability', 'energy', 'speechiness',
                  'acousticness', 'instrumentalness', 'liveness', 'valence']

for feat in audio_features:
    fuera = df[(df[feat] < 0) | (df[feat] > 1)][feat].count()
    if fuera > 0:
        print(f'{feat}: {fuera} valores fuera del rango [0, 1]')
    else:
        print(f'{feat}: OK')

danceability: OK
energy: OK
speechiness: OK
acousticness: OK
instrumentalness: OK
liveness: OK
valence: OK


# Renombramiento de Columnas

In [5]:
# Normalizar loudness y tempo antes de seleccionar
df['loudness_norm'] = (df['loudness'] - df['loudness'].min()) / (df['loudness'].max() - df['loudness'].min())
df['tempo_norm'] = (df['tempo'] - df['tempo'].min()) / (df['tempo'].max() - df['tempo'].min())

# Columnas que nos interesan
# ITEM_ID es obligatorio para Amazon Personalize

items = df[[
    'track_id',           # ITEM_ID
    'track_name',         # titulo
    'artists',            # artista
    'album_name',         # album
    'track_genre',        # genero
    'popularity',         # popularidad
    'duration_ms',        # duracion_ms
    'explicit',           # explicito
    # Features de audio para Content-Based Filtering
    'danceability',
    'energy',
    'speechiness',
    'acousticness',
    'instrumentalness',
    'liveness',
    'valence',
    'loudness_norm',
    'tempo_norm'
]].copy()

# Renombrar para consistencia con el esquema del proyecto
items = items.rename(columns={
    'track_id'    : 'ITEM_ID',
    'track_name'  : 'titulo',
    'artists'     : 'artista',
    'album_name'  : 'album',
    'track_genre' : 'genero',
    'popularity'  : 'popularidad',
    'duration_ms' : 'duracion_ms',
    'explicit'    : 'explicito'
})

print(f'Shape final de items: {items.shape}')
items.head()

KeyError: "['loudness_norm', 'tempo_norm'] not in index"